## Что изменилось

| | Оригинал | Supervisor pattern |
|---|---|---|
| **ManagerAgent** | Строит JSON-план → отдаёт pipeline | Supervisor: вызывает агентов как `@tool`, проверяет результат каждого |
| **Pipeline** | Итерирует по `ctx.plan` детерминированно | Просто запускает `ManagerAgent.run()` — больше ничего |
| **Роутинг** | Статичный (порядок из плана) | Динамичный (LLM решает после каждого шага) |
| **Проверка** | Нет | Supervisor получает `ToolMessage` с результатом и валидирует его |
| **`ctx.plan`** | Нужен для роутинга | Убран — роутинг внутри Supervisor |
| **`@agent_step`** | Обновлял `ctx.plan` | Убран — логирование прямо в `run()` |
| **Worker-агенты** | Без изменений | Без изменений (DDD/TDD/Developer/Reviewer/Executor) |

## Импорты

In [3]:
import os
import json
import shutil
import subprocess
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from dotenv import load_dotenv
from pathlib import Path
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage

load_dotenv()

ModuleNotFoundError: No module named 'dotenv'

## LLMClient

добавлен `get_raw_client()` — нужен Supervisor для `bind_tools()`.

In [2]:
class LLMClient:
    def __init__(self):
        self._client = ChatOpenAI(
            base_url=os.getenv("OPENAI_API_HOST"),
            api_key=os.getenv("OPENAI_API_KEY"),
            model="openai/gpt-oss-20b",
            timeout=None,
            temperature=0.7,
        )

    def generate(self, prompt: str) -> str:
        return self._client.invoke(prompt).content

    def get_raw_client(self) -> ChatOpenAI:
        """Возвращает нативный клиент — нужен Supervisor для bind_tools()."""
        return self._client

## AgentContext — общая память

убрано поле `plan` — роутинг теперь внутри Supervisor, план не нужен.
Добавлен метод `status()` — Supervisor читает его после каждого tool-call.

In [3]:
@dataclass
class AgentContext:
    requirements: str
    # план убран — Supervisor сам управляет порядком через tool-calling
    domain: str | None = None      
    tests:  str | None = None       
    review: str | None = None    
    files:  dict[str, str] = field(default_factory=dict) 

    def status(self) -> str:
        """Срез состояния для Supervisor: что уже сделано, что ещё нет."""
        return json.dumps({
            "domain_done":    self.domain   is not None,
            "tests_done":     self.tests    is not None,
            "developer_done": bool(self.files),
            "review_done":    self.review   is not None,
            "files_on_disk":  list(self.files.keys()),
        }, ensure_ascii=False)

## Tools (без изменений)

In [4]:
@tool
def run_docker_compose(compose_path: str = "repo/docker-compose.yml") -> str:
    """Run docker compose up --build -d for given compose file path."""
    command = f"docker compose -f {compose_path} up --build -d".split()
    result = subprocess.run(command, capture_output=True, text=True)
    output = result.stdout + result.stderr
    if result.returncode == 0:
        return f"✅ Docker launched.\n{output[:500]}"
    return f"❌ Docker failed (code {result.returncode}):\n{output[:500]}"


@tool
def check_docker_status() -> str:
    """Check running docker containers."""
    result = subprocess.run(["docker", "ps"], capture_output=True, text=True)
    return result.stdout


@tool
def write_output_files(files_json: str) -> str:
    """Write only .html, .js, .csv files to OUTPUT_DIR. Everything else is skipped."""
    output_dir = Path(os.getenv("OUTPUT_DIR", "../scripts/demo_02"))
    shutil.rmtree(output_dir, ignore_errors=True)
    output_dir.mkdir(parents=True, exist_ok=True)
    files = json.loads(files_json)
    written = []
    for filename, content in files.items():
        ext = Path(filename).suffix.lower()
        if ext in (".html", ".js", ".csv"):
            (output_dir / filename).write_text(content, encoding="utf-8")
            written.append(filename)
        else:
            print(f"  ⏭ Skipped: {filename}")
    return f"Written to {output_dir}: {written}"

## BaseAgent (без изменений)

In [5]:
class BaseAgent(ABC):
    def __init__(self, llm: LLMClient):
        self.llm = llm

    @abstractmethod
    def run(self, ctx: AgentContext) -> AgentContext:
        ...

## Agents

Убран декоратор `@agent_step` (он обновлял `ctx.plan`, которого больше нет).
Логирование `▶ started` / `✅ done` перенесено напрямую в `run()`.

In [6]:
# ===== 1. DDD Agent =====

class DDDAgent(BaseAgent):

    def run(self, ctx: AgentContext) -> AgentContext:
        print("\n▶ [DDD] started")
        prompt = f"""
Ты аналитик доменных моделей.
Опиши доменную модель для веб-приложения по требованиям ниже.

Требования: {ctx.requirements}

Включи:
- Основные сущности и их атрибуты
- Связи между сущностями
- Ключевые бизнес-правила

Отвечай структурированно и кратко — это будет использовано для генерации кода.
"""
        ctx.domain = self.llm.generate(prompt)
        print("  Domain model → ctx.domain (in memory)")
        print("✅ [DDD] done")
        return ctx

In [7]:
# ===== 2. TDD Agent =====

class TDDAgent(BaseAgent):

    def run(self, ctx: AgentContext) -> AgentContext:
        print("\n▶ [TDD] started")
        prompt = f"""
Ты инженер по тестированию.
Напиши pytest-тесты для веб-приложения.

Требования: {ctx.requirements}
Доменная модель: {ctx.domain}

Правила:
- Покрой основную бизнес-логику
- Используй только стандартные библиотеки Python и pytest
- Верни только Python-код без markdown и пояснений
"""
        ctx.tests = self.llm.generate(prompt)
        print("  Tests → ctx.tests (in memory)")
        print("✅ [TDD] done")
        return ctx

In [8]:
# ===== 3. Developer Agent =====

class DeveloperAgent(BaseAgent):

    def run(self, ctx: AgentContext) -> AgentContext:
        print("\n▶ [Developer] started")

        html_prompt = f"""
Ты frontend-разработчик. Создай одностраничное веб-приложение.

Требования: {ctx.requirements}
Доменная модель: {ctx.domain}

СТРОГИЕ ПРАВИЛА:
1. Верни ТОЛЬКО валидный HTML от <!DOCTYPE html> до </html>
2. Без markdown, без пояснений, без блоков ```
3. Все стили — внутри <style> в <head>. Дизайн: современный, чистый, адаптивный
4. Все интерактивные элементы должны иметь id или class — для управления из script.js
5. Если приложение предполагает графики — добавь:
   <script src="https://cdn.jsdelivr.net/npm/chart.js@4.3.3/dist/chart.umd.min.js"></script>
   и <canvas id="mainChart"></canvas> в нужном месте
6. В конце <body> подключи: <script src="script.js"></script>
7. Структура и содержимое страницы должны точно соответствовать требованиям
8. Все теги закрыты. HTML валиден.
"""
        js_prompt = f"""
Ты JavaScript-разработчик. Создай script.js для веб-приложения.

Требования: {ctx.requirements}
Доменная модель: {ctx.domain}

СТРОГИЕ ПРАВИЛА:
1. Верни ТОЛЬКО чистый JS-код без markdown и блоков ```
2. Весь код оберни в: document.addEventListener('DOMContentLoaded', () => {{ /* весь код */ }})
3. Создай демо-данные прямо в коде — массивы объектов, соответствующие доменной модели
   Данные должны быть реалистичными и соответствовать типу приложения
   Минимум 8-10 записей
4. Для вставки данных в DOM используй createElement + textContent (не innerHTML с данными)
5. Реализуй всю интерактивность из требований: фильтры, сортировка, поиск, кнопки и т.д.
6. Если в HTML есть canvas#mainChart — создай Chart.js график подходящего типа
   (Chart уже подключён глобально, не импортируй его)
7. Не используй fetch, axios или внешние запросы — только локальные данные
8. Код должен работать сразу после открытия index.html в браузере
"""
        csv_prompt = f"""
Создай CSV-файл с демонстрационными данными для веб-приложения.

Требования: {ctx.requirements}
Доменная модель: {ctx.domain}

ПРАВИЛА:
1. Верни ТОЛЬКО CSV-текст без markdown и блоков ```
2. Первая строка — заголовки колонок, соответствующие доменной модели
3. Минимум 20 строк реалистичных данных
4. Форматы: даты YYYY-MM-DD, числа без пробелов, строки без лишних кавычек
"""
        print("  Generating HTML...")
        index_html = self.llm.generate(html_prompt)
        print("  Generating JS...")
        script_js = self.llm.generate(js_prompt)
        print("  Generating CSV...")
        data_csv = self.llm.generate(csv_prompt)

        ctx.files = {
            "index.html": index_html,
            "script.js":  script_js,
            "data.csv":   data_csv,
        }
        result = write_output_files.invoke({"files_json": json.dumps(ctx.files, ensure_ascii=False)})
        print(f"  {result}")
        print("✅ [Developer] done")
        return ctx

In [9]:
# ===== 4. Reviewer Agent =====

class ReviewerAgent(BaseAgent):

    def run(self, ctx: AgentContext) -> AgentContext:
        print("\n▶ [Reviewer] started")
        html_snippet = ctx.files.get("index.html", "")[:1200]
        js_snippet   = ctx.files.get("script.js",  "")[:1200]
        prompt = f"""
Ты senior-разработчик. Проведи code review сгенерированного веб-приложения.

Требования к приложению: {ctx.requirements}

HTML (фрагмент):
{html_snippet}

JS (фрагмент):
{js_snippet}

Найди конкретные проблемы и предложи улучшения (до 10 пунктов).
Формат каждого пункта: номер. Проблема → Решение
"""
        ctx.review = self.llm.generate(prompt)
        print("  Review → ctx.review (in memory)")
        print("✅ [Reviewer] done")
        return ctx

In [10]:
# ===== 5. Executor Agent =====

class ExecutorAgent(BaseAgent):

    def run(self, ctx: AgentContext) -> AgentContext:
        print("\n▶ [Executor] started")
        compose_path = "../scripts/docker-compose.yml"
        if not Path(compose_path).exists():
            print(f"  ⚠ {compose_path} not found, skipping docker launch")
            print("✅ [Executor] done")
            return ctx
        result = run_docker_compose.invoke({"compose_path": compose_path})
        print(f"  {result[:300]}")
        print("✅ [Executor] done")
        return ctx

## ManagerAgent — Supervisor 

### Три ключевых механизма

**1. Workers как `@tool`** — каждый агент обёрнут в функцию с описанием.
LLM видит описание и решает кого вызвать.

**2. Validation после каждого tool-call** — `_validate()` проверяет:
- результат не пустой
- содержит ожидаемые ключевые слова
- файлы реально записаны на диск

**3. `ToolMessage` с итогом проверки** — LLM получает `✅ OK` или `⚠ RETRY: причина`
и сам решает: перейти к следующему агенту или повторить текущий.

```
Supervisor loop:

  messages = [SystemMessage, HumanMessage(requirements)]
  
  while iterations < MAX:
      response = llm_with_tools.invoke(messages)
      
      if response.tool_calls:              # LLM вызывает агента
          result  = worker.run(ctx)        # запускаем worker
          verdict = _validate(name, ctx)   # проверяем результат
          messages.append(ToolMessage(verdict))  # говорим LLM что получилось
      else:
          break                            # LLM решил: все агенты выполнены
```

In [11]:
# ===== ManagerAgent — Supervisor =====

class ManagerAgent(BaseAgent):
    """
    Supervisor-агент: управляет воркерами через LLM tool-calling.

    Цикл работы:
      1. LLM получает список tools и системный промт с порядком запуска
      2. На каждой итерации LLM выбирает следующий tool (агента)
      3. Supervisor запускает worker, проверяет результат
      4. Результат проверки идёт обратно в LLM как ToolMessage
      5. Когда все агенты выполнены — LLM не делает tool_calls → loop завершается
    """

    MAX_ITERATIONS = 10  # защита от бесконечного цикла

    def __init__(self, llm: LLMClient):
        super().__init__(llm)
        self._ctx: AgentContext | None = None  # общая память, доступна всем workers

        # ── Оборачиваем каждый worker в @tool ───────────────────────────────
        # LLM читает docstring каждого tool и решает какой вызвать следующим.
        # После вызова tool возвращает строку — LLM видит её как результат шага.
        
        @tool
        def run_plan() -> str:
            """Агент, который разрабатывает план разработки."""
            self._ctx = DDDAgent(self.llm).run(self._ctx)
            # Проверяем что доменная модель реально сгенерирована
            ok = self._ctx.domain is not None and len(self._ctx.domain) > 100
            return "✅ DDD OK" if ok else "⚠ DDD FAILED: domain пустой"

        @tool
        def run_ddd() -> str:
            """Агент, который строит доменную модель."""
            self._ctx = DDDAgent(self.llm).run(self._ctx)
            # Проверяем что доменная модель реально сгенерирована
            ok = self._ctx.domain is not None and len(self._ctx.domain) > 100
            return "✅ DDD OK" if ok else "⚠ DDD FAILED: domain пустой"

        @tool
        def run_tdd() -> str:
            """Агент, который создает функциональные тесты на основе дизайна доменной модели."""
            self._ctx = TDDAgent(self.llm).run(self._ctx)
            # Проверяем что в коде есть хотя бы одна тестовая функция
            ok = self._ctx.tests is not None and "def test_" in self._ctx.tests
            return "✅ TDD OK" if ok else "⚠ TDD FAILED: тесты не сгенерированы"

        @tool
        def run_developer() -> str:
            """
            Агент, который реализует приложение на основе требований доменной области. ...
            """
            self._ctx = DeveloperAgent(self.llm).run(self._ctx)
            # Проверяем что оба ключевых файла записаны на диск
            ok = "index.html" in self._ctx.files and "script.js" in self._ctx.files
            return "✅ Developer OK" if ok else "⚠ Developer FAILED: файлы не созданы"

        @tool
        def run_reviewer() -> str:
            """
            Агент, который проводит ревью.
            """
            self._ctx = ReviewerAgent(self.llm).run(self._ctx)
            # Проверяем что review содержательный, а не пустая строка
            ok = self._ctx.review is not None and len(self._ctx.review) > 50
            return "✅ Reviewer OK" if ok else "⚠ Reviewer FAILED: review пустой"

        @tool
        def run_executor() -> str:
            """
            Агент, который запускает docker-compose.
            """
            self._ctx = ExecutorAgent(self.llm).run(self._ctx)
            # Docker может отсутствовать — это не ошибка, всегда OK
            return "✅ Executor OK"

        # Список tools передаём в LLM через bind_tools —
        # так LLM знает какие инструменты доступны и может делать tool_calls
        self._tools      = [run_ddd, run_tdd, run_developer, run_reviewer, run_executor]
        self._tool_map   = {t.name: t for t in self._tools}  # имя → tool для быстрого поиска
        self._llm_with_tools = self.llm.get_raw_client().bind_tools(self._tools)

    def run(self, ctx: AgentContext) -> AgentContext:
        self._ctx = ctx

        messages = [
            SystemMessage(content=(
                "Ты Supervisor мультиагентной системы.\n"
                "Запускай агентов строго по порядку через tool-calls:\n"
                "run_ddd → run_tdd → run_developer → run_reviewer → run_executor\n"
                "Каждый агент — ровно 1 раз. "
                "Когда все выполнены — не делай tool_calls, верни финальный отчёт."
            )),
            HumanMessage(content=f"Требования: {ctx.requirements}"),
        ]

        print("\n▶ [Supervisor] started")

        for i in range(self.MAX_ITERATIONS):
            print(f"\n  [Supervisor] iteration {i + 1}/{self.MAX_ITERATIONS}")

            # LLM смотрит на историю сообщений и решает:
            # сделать tool_call (запустить агента) или вернуть финальный текст
            response = self._llm_with_tools.invoke(messages)
            messages.append(response)  # добавляем ответ LLM в историю

            # Нет tool_calls → LLM решил что все агенты выполнены
            if not response.tool_calls:
                print("✅ [Supervisor] all agents done")
                break

            # Есть tool_calls → запускаем каждый
            for tc in response.tool_calls:
                print(f"{tc=}")

                # Нормализация имени: некоторые модели добавляют суффиксы
                # например run_tdd<|channel|>commentary → берём только run_tdd
                name = tc["name"].split("<|")[0].split("::")[0].strip()

                if tc["name"] != name:
                    print(f"  ⚠ tool name normalized: '{tc['name']}' → '{name}'")

                print(f"  → tool_call: {name}")

                if name in self._tool_map:
                    # Вызываем worker — он пишет результат в self._ctx
                    # и возвращает строку ✅/⚠ для LLM
                    result = self._tool_map[name].invoke({**tc, "name": name})
                else:
                    result = f"⚠ Unknown tool: '{name}'. Используй: {list(self._tool_map)}"

                # ToolMessage — LLM увидит результат на следующей итерации
                # и решит что делать дальше
                messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

        return self._ctx

## Pipeline

pipeline больше не итерирует по `ctx.plan`.
Он просто создаёт контекст и передаёт управление Supervisor'у — вся логика внутри него.

In [12]:
# def run_pipeline(prompt: str, llm: LLMClient | None = None) -> AgentContext:
#     if llm is None:
#         llm = LLMClient()

#     ctx = AgentContext(requirements=prompt)

#     # Supervisor сам роутит агентов и проверяет результаты
#     ctx = ManagerAgent(llm).run(ctx)

#     print("\n Pipeline complete!")
#     print(f"   Files on disk : {list(ctx.files.keys())}")
#     print(f"   Domain in mem : {bool(ctx.domain)}")
#     print(f"   Tests  in mem : {bool(ctx.tests)}")
#     print(f"   Review in mem : {bool(ctx.review)}")
#     return ctx

## Запуск

In [13]:
prompt = "Создай интерактивный дашборд с таблицей данных и графиком продаж."
ctx = run_pipeline(prompt)


▶ [Supervisor] started

  [Supervisor] iteration 1/10
tc={'name': 'run_ddd', 'args': {}, 'id': 'chatcmpl-tool-86d8b0d4f567f073', 'type': 'tool_call'}
  → tool_call: run_ddd

▶ [DDD] started
  Domain model → ctx.domain (in memory)
✅ [DDD] done

  [Supervisor] iteration 2/10
tc={'name': 'run_tdd<|channel|>commentary', 'args': {}, 'id': 'chatcmpl-tool-b3d7819ac24cceff', 'type': 'tool_call'}
  ⚠ tool name normalized: 'run_tdd<|channel|>commentary' → 'run_tdd'
  → tool_call: run_tdd

▶ [TDD] started
  Tests → ctx.tests (in memory)
✅ [TDD] done

  [Supervisor] iteration 3/10
tc={'name': 'run_developer<|channel|>commentary', 'args': {}, 'id': 'chatcmpl-tool-8a40c05449a00bd7', 'type': 'tool_call'}
  ⚠ tool name normalized: 'run_developer<|channel|>commentary' → 'run_developer'
  → tool_call: run_developer

▶ [Developer] started
  Generating HTML...
  Generating JS...
  Generating CSV...
  Written to ../scripts/demo_03: ['index.html', 'script.js', 'data.csv']
✅ [Developer] done

  [Supervisor]